In [1]:
# from paddleocr import PPStructureV3

In [2]:
# pipeline = PPStructureV3(
#     # engine="transformers", #intelegent model is used 
#     use_formula_recognition=False, #it will not detect foumual 
#     wired_table_structure_recognition_model_name="RT-DETR-L_wired_table_cell_det", # this model is used to detect the content without table lines 
#     lang="en",
#     use_doc_orientation_classify=False,
#     use_doc_unwarping=False,
#     use_textline_orientation=False,
#     device=paddle.device.get_device(),
#     cpu_threads=8 
# )


In [3]:
# output = pipeline.predict(img)

In [4]:
# output

In [5]:
# !nvcc --version

In [6]:
# !nvidia-smi

In [7]:
# print("CUDA built:", paddle.device.is_compiled_with_cuda())
# print("GPU count:", paddle.device.cuda.device_count())
# print("Device:", paddle.device.get_device())

# display version of python used 

In [69]:
!py --version

Python 3.10.8


# import the libary

In [9]:
from paddleocr import PaddleOCR 
import paddle
import cv2
import numpy as np
import os
from llama_cpp import Llama
import json

D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
D:\Anuj_S_Jain\Data\Code\Project\Society_MCE_QT\Society_MCE\Prototype\.venv\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccac

# device used

In [70]:
paddle.device.get_device()

'cpu'

In [71]:
paddle.device.cuda.device_count()

0

# loading image path

In [72]:
i="4"
image_path = f"D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/{i}.jpg"
image_path

'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/training data/diff_img/4.jpg'

# read img

In [73]:
print(os.path.exists(image_path))

True


In [74]:
img = cv2.imread(image_path)

# run paddleocr on img 

In [ ]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
    # setting tread 
    # os.environ["OMP_NUM_THREADS"] = "8"
    # os.environ["MKL_NUM_THREADS"] = "8"
    # applying ocr on the image
    ocr = PaddleOCR(use_doc_orientation_classify=False, 
        use_doc_unwarping=False, 
        use_textline_orientation=False,
        lang='en',
        device=paddle.device.get_device(),
        cpu_threads=8 
       )
    result = ocr.predict(img)

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Anuj S Jain\.paddlex\official_models\PP-OCRv5_server_det`.


In [15]:
result

NameError: name 'result' is not defined

# extracting the table from the img 

In [52]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
   
    # converting img to gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # cv2.imwrite(f'./output/opencv/gray{i}.jpeg', gray)

    # splits the image and clahe 
    # C – Contrast
    # L – Limited
    # A – Adaptive
    # H – Histogram
    # E – Equalization
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray_clahe = clahe.apply(gray)    
    # cv2.imwrite(f'./output/opencv/gray_clahe{i}.jpeg', gray_clahe)
    
    # Better threshold (adaptive handles uneven lighting)
    thresh = cv2.adaptiveThreshold(
        gray_clahe, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15, 10
    )
     
    
    # --- Horizontal lines ---
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (60,1))
    horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
    cv2.imwrite(f'./output/opencv/horizontal{i}.jpeg', horizontal)
    
    # connect broken horizontal lines
    horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=3)
    cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)
    
    # --- Vertical lines ---
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,60))
    vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
    # cv2.imwrite(f'./output/opencv/vertical{i}.jpeg', vertical)
     
    # connect broken vertical lines
    vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=3)
    
    cv2.imwrite(f'./output/opencv/vertical_dilate{i}.jpeg', vertical)
    h_img_path=f'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/output/opencv/horizontal_dilated{i}.jpeg'
    h_img = cv2.imread(h_img_path)

In [53]:
# horizontal[100][:]=126
# horizontal[110][:]=126
# horizontal[120][:]=126
# horizontal[130][:]=126
# cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)

In [54]:
# import os
# ServerApp.iopub_data_rate_limit=1000000.0 
# ServerApp.rate_limit_window=3.0 
# !jupyter lab --ServerApp.iopub_data_rate_limit=9000000.0 
# !jupyter lab --ServerApp.rate_limit_window=10.0 
# print(os.path.exists(h_img_path))
# print(horizontal.tolist())

In [113]:
def get_vh_lines(img,image_path):
    if (img is None) :
        print(f"{image_path} this is incorrect no img found")
    else:
       
        # converting img to gray
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        # cv2.imwrite(f'./output/opencv/gray{i}.jpeg', gray)
    
        # splits the image and clahe 
        # C – Contrast
        # L – Limited
        # A – Adaptive
        # H – Histogram
        # E – Equalization
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        gray_clahe = clahe.apply(gray)    
        # cv2.imwrite(f'./output/opencv/gray_clahe{i}.jpeg', gray_clahe)
        
        # Better threshold (adaptive handles uneven lighting)
        thresh = cv2.adaptiveThreshold(
            gray_clahe, 255,
            cv2.ADAPTIVE_THRESH_MEAN_C,
            cv2.THRESH_BINARY_INV,
            15, 10
        )
         
        
        # --- Horizontal lines ---
        horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (60,1))
        horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
        # cv2.imwrite(f'./output/opencv/horizontal{i}.jpeg', horizontal)
        
        # connect broken horizontal lines
        horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=3)
        # cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)
        
        # --- Vertical lines ---
        vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,60))
        vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
        # cv2.imwrite(f'./output/opencv/vertical{i}.jpeg', vertical)
         
        # connect broken vertical lines
        vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=3)
        
        # cv2.imwrite(f'./output/opencv/vertical_dilate{i}.jpeg', vertical)
        # h_img_path=f'D:/Anuj_S_Jain/Data/Code/Project/Society_MCE_QT/Society_MCE/Prototype/output/opencv/horizontal_dilated{i}.jpeg'
        # h_img = cv2.imread(h_img_path)

        return horizontal,vertical

def get_h_counters(horizontal):
    contours_h, _ = cv2.findContours(horizontal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return contours_h
def get_v_counters(vertical):
    contours_v, _ = cv2.findContours(vertical, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return contours_v

def get_h_angle(contours_h):
    angle_arr=[]
    for counter in contours_h:
        vx, vy, x0, y0 = cv2.fitLine(counter, cv2.DIST_L2, 0, 0.01, 0.01)
        angle = np.arctan2(vy, vx)
        rotation_angle =np.degrees(angle)
        angle_arr.append(rotation_angle)
    return angle_arr
    
def get_v_angle(contours_v):
    angle_arr=[]
    for counter in contours_v:
        vx, vy, x0, y0 = cv2.fitLine(counter, cv2.DIST_L2, 0, 0.01, 0.01)
        angle = np.arctan2(vy, vx)
        rotation_angle =np.degrees(angle)
        angle_arr.append(rotation_angle)
    return angle_arr

def rotate_angle_avg(angle_arr):
    rotation_angle_avg=float(sum(angle_arr)/len(angle_arr))
    return rotation_angle_avg

def rotate_img(img,rotation_angle_avg):
    (h, w) = img.shape[:2]
    center = (w // 2, h // 2)
    img_matrix = cv2.getRotationMatrix2D(center, rotation_angle_avg, 1.0)
    rotated_img = cv2.warpAffine(img, img_matrix, (w, h))
    return rotated_img


    
    

In [116]:
def rotation_correct_img(img,image_path):
    horizontal,vertical=get_vh_lines(img,image_path)
    if len(horizontal) >= len(vertical):
        contours_h=get_h_counters(horizontal)
        angle_arr_h=get_h_angle(contours_h)
        rotation_angle_avg=rotate_angle_avg(angle_arr_h)
        rotated_img=rotate_img(img,rotation_angle_avg)
        return rotated_img
    elif len(horizontal) < len(vertical):
        contours_v=get_v_counters(vertical)
        angle_arr_v=get_v_angle(contours_v)
        rotation_angle_avg=rotate_angle_avg(angle_arr_v)
        rotated_img=rotate_img(img,-rotation_angle_avg)
        return rotated_img
    elif len(horizontal) == 0 or len(vertical):
        pass
rotated_img=rotation_correct_img(img,image_path)
cv2.imwrite(f'./output/opencv/rotated_img_final{i}.jpeg', rotated_img)

C:\Users\Anuj S Jain\AppData\Local\Temp\ipykernel_30780\2422193211.py:78: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  rotation_angle_avg=float(sum(angle_arr)/len(angle_arr))


True

In [26]:
angle_arr=[]
for contour in contours:
    vx, vy, x0, y0 = cv2.fitLine(contour, cv2.DIST_L2, 0, 0.01, 0.01)
    angle = np.arctan2(vy, vx)
    rotation_angle =np.degrees(angle)
    angle_arr.append(rotation_angle)
    
angel_avg=sum(angle_arr)/len(angle_arr)
angel_avg

array([-0.48301023], dtype=float32)

In [101]:
 # Find contours
contours_h, _ = cv2.findContours(horizontal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contours_v, _ = cv2.findContours(vertical, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# result = cv2.drawContours(h_img, contours[1], -1, (0, 255, 0), 2)

# cv2.imwrite(f'./output/opencv/output---{i}.jpeg', result)

In [ ]:
len(contours[1])

In [102]:
  vxh, vyh, x0h, y0h = cv2.fitLine(contours_h[1], cv2.DIST_L2, 0, 0.01, 0.01)
  vxv, vyv, x0v, y0v = cv2.fitLine(contours_v[1], cv2.DIST_L2, 0, 0.01, 0.01)

In [103]:
angle_arr

NameError: name 'angle_arr' is not defined

In [107]:
# angle = np.arctan2(vy, vx)
angle = np.arctan2(vxv, vyv)
angle

array([-4.371139e-08], dtype=float32)

In [108]:
rotation_angle =float( np.degrees(angle))
rotation_angle

C:\Users\Anuj S Jain\AppData\Local\Temp\ipykernel_30780\1476383597.py:1: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  rotation_angle =float( np.degrees(angle))


-2.504477834008867e-06

In [104]:
(h, w) = img.shape[:2]
center = (w // 2, h // 2)

M = cv2.getRotationMatrix2D(center, rotation_angle, 1.0)
M = cv2.getRotationMatrix2D(center, -rotation_angle, 1.0) #1.0 no change to img is made
rotated = cv2.warpAffine(img, M, (w, h))#m tell how to move the points 

cv2.imwrite(f'./output/opencv/rotated{i}.jpeg', rotated)

True

In [100]:
contours_arr = list(contours)
contours_arr

[array([[[1399, 1057]],
 
        [[1399, 1128]],
 
        [[1400, 1129]],
 
        [[1400, 1191]],
 
        [[1401, 1192]],
 
        [[1401, 1229]],
 
        [[1402, 1230]],
 
        [[1402, 1301]],
 
        [[1403, 1302]],
 
        [[1403, 1345]],
 
        [[1404, 1346]],
 
        [[1404, 1371]],
 
        [[1405, 1372]],
 
        [[1405, 1414]],
 
        [[1406, 1415]],
 
        [[1409, 1415]],
 
        [[1409, 1324]],
 
        [[1408, 1323]],
 
        [[1408, 1285]],
 
        [[1407, 1284]],
 
        [[1407, 1214]],
 
        [[1406, 1213]],
 
        [[1406, 1175]],
 
        [[1405, 1174]],
 
        [[1405, 1121]],
 
        [[1404, 1120]],
 
        [[1404, 1069]],
 
        [[1403, 1068]],
 
        [[1403, 1057]]], dtype=int32),
 array([[[   0,  755]],
 
        [[   0,  820]],
 
        [[   1,  821]],
 
        [[   1,  897]],
 
        [[   2,  898]],
 
        [[   2,  931]],
 
        [[   3,  932]],
 
        [[   3, 1003]],
 
        [[   4, 1004]],
 

In [36]:
contours_x=sorted(contours_arr[1], key=lambda p: p[0][0])
contours_y=sorted(contours_arr[1], key=lambda p: p[0][1])


In [ ]:
for counter_pnt  in contours_x:
    

In [39]:
contours_arr[1]

array([[[  36, 1534]],

       [[  36, 1539]],

       [[ 136, 1539]],

       [[ 137, 1540]],

       [[ 223, 1540]],

       [[ 224, 1541]],

       [[ 352, 1541]],

       [[ 353, 1542]],

       [[ 541, 1542]],

       [[ 542, 1543]],

       [[ 565, 1543]],

       [[ 566, 1544]],

       [[ 769, 1544]],

       [[ 770, 1545]],

       [[1004, 1545]],

       [[1005, 1546]],

       [[1050, 1546]],

       [[1051, 1547]],

       [[1144, 1547]],

       [[1145, 1548]],

       [[1218, 1548]],

       [[1219, 1549]],

       [[1263, 1549]],

       [[1264, 1550]],

       [[1332, 1550]],

       [[1333, 1551]],

       [[1374, 1551]],

       [[1375, 1552]],

       [[1429, 1552]],

       [[1430, 1553]],

       [[1520, 1553]],

       [[1521, 1554]],

       [[1541, 1554]],

       [[1542, 1555]],

       [[1609, 1555]],

       [[1610, 1556]],

       [[1620, 1556]],

       [[1621, 1557]],

       [[1666, 1557]],

       [[1667, 1558]],

       [[1717, 1558]],

       [[1718, 1

In [38]:
contours_x

[array([[  36, 1534]], dtype=int32),
 array([[  36, 1539]], dtype=int32),
 array([[ 126, 1534]], dtype=int32),
 array([[ 127, 1535]], dtype=int32),
 array([[ 136, 1539]], dtype=int32),
 array([[ 137, 1540]], dtype=int32),
 array([[ 223, 1540]], dtype=int32),
 array([[ 224, 1541]], dtype=int32),
 array([[ 230, 1535]], dtype=int32),
 array([[ 231, 1536]], dtype=int32),
 array([[ 328, 1536]], dtype=int32),
 array([[ 329, 1537]], dtype=int32),
 array([[ 352, 1541]], dtype=int32),
 array([[ 353, 1542]], dtype=int32),
 array([[ 457, 1537]], dtype=int32),
 array([[ 458, 1538]], dtype=int32),
 array([[ 541, 1542]], dtype=int32),
 array([[ 542, 1543]], dtype=int32),
 array([[ 565, 1543]], dtype=int32),
 array([[ 566, 1544]], dtype=int32),
 array([[ 603, 1538]], dtype=int32),
 array([[ 604, 1539]], dtype=int32),
 array([[ 753, 1539]], dtype=int32),
 array([[ 754, 1540]], dtype=int32),
 array([[ 769, 1544]], dtype=int32),
 array([[ 770, 1545]], dtype=int32),
 array([[ 926, 1540]], dtype=int32),
 

In [37]:
contours_y

[array([[  36, 1534]], dtype=int32),
 array([[ 126, 1534]], dtype=int32),
 array([[ 230, 1535]], dtype=int32),
 array([[ 127, 1535]], dtype=int32),
 array([[ 328, 1536]], dtype=int32),
 array([[ 231, 1536]], dtype=int32),
 array([[ 457, 1537]], dtype=int32),
 array([[ 329, 1537]], dtype=int32),
 array([[ 603, 1538]], dtype=int32),
 array([[ 458, 1538]], dtype=int32),
 array([[  36, 1539]], dtype=int32),
 array([[ 136, 1539]], dtype=int32),
 array([[ 753, 1539]], dtype=int32),
 array([[ 604, 1539]], dtype=int32),
 array([[ 137, 1540]], dtype=int32),
 array([[ 223, 1540]], dtype=int32),
 array([[ 926, 1540]], dtype=int32),
 array([[ 754, 1540]], dtype=int32),
 array([[ 224, 1541]], dtype=int32),
 array([[ 352, 1541]], dtype=int32),
 array([[1064, 1541]], dtype=int32),
 array([[ 927, 1541]], dtype=int32),
 array([[ 353, 1542]], dtype=int32),
 array([[ 541, 1542]], dtype=int32),
 array([[1129, 1542]], dtype=int32),
 array([[1065, 1542]], dtype=int32),
 array([[ 542, 1543]], dtype=int32),
 

In [27]:
for i in contours[1].tolist():
    print(i)

[[36, 1534]]
[[36, 1539]]
[[136, 1539]]
[[137, 1540]]
[[223, 1540]]
[[224, 1541]]
[[352, 1541]]
[[353, 1542]]
[[541, 1542]]
[[542, 1543]]
[[565, 1543]]
[[566, 1544]]
[[769, 1544]]
[[770, 1545]]
[[1004, 1545]]
[[1005, 1546]]
[[1050, 1546]]
[[1051, 1547]]
[[1144, 1547]]
[[1145, 1548]]
[[1218, 1548]]
[[1219, 1549]]
[[1263, 1549]]
[[1264, 1550]]
[[1332, 1550]]
[[1333, 1551]]
[[1374, 1551]]
[[1375, 1552]]
[[1429, 1552]]
[[1430, 1553]]
[[1520, 1553]]
[[1521, 1554]]
[[1541, 1554]]
[[1542, 1555]]
[[1609, 1555]]
[[1610, 1556]]
[[1620, 1556]]
[[1621, 1557]]
[[1666, 1557]]
[[1667, 1558]]
[[1717, 1558]]
[[1718, 1559]]
[[1776, 1559]]
[[1777, 1560]]
[[1804, 1560]]
[[1805, 1561]]
[[1835, 1561]]
[[1836, 1562]]
[[1916, 1562]]
[[1916, 1559]]
[[1909, 1559]]
[[1908, 1558]]
[[1880, 1558]]
[[1879, 1557]]
[[1847, 1557]]
[[1846, 1556]]
[[1794, 1556]]
[[1793, 1555]]
[[1763, 1555]]
[[1762, 1554]]
[[1720, 1554]]
[[1719, 1553]]
[[1676, 1553]]
[[1675, 1552]]
[[1618, 1552]]
[[1617, 1551]]
[[1583, 1551]]
[[1582, 155

In [19]:
contours[1].tolist()

[[[36, 1534]],
 [[36, 1539]],
 [[136, 1539]],
 [[137, 1540]],
 [[223, 1540]],
 [[224, 1541]],
 [[352, 1541]],
 [[353, 1542]],
 [[541, 1542]],
 [[542, 1543]],
 [[565, 1543]],
 [[566, 1544]],
 [[769, 1544]],
 [[770, 1545]],
 [[1004, 1545]],
 [[1005, 1546]],
 [[1050, 1546]],
 [[1051, 1547]],
 [[1144, 1547]],
 [[1145, 1548]],
 [[1218, 1548]],
 [[1219, 1549]],
 [[1263, 1549]],
 [[1264, 1550]],
 [[1332, 1550]],
 [[1333, 1551]],
 [[1374, 1551]],
 [[1375, 1552]],
 [[1429, 1552]],
 [[1430, 1553]],
 [[1520, 1553]],
 [[1521, 1554]],
 [[1541, 1554]],
 [[1542, 1555]],
 [[1609, 1555]],
 [[1610, 1556]],
 [[1620, 1556]],
 [[1621, 1557]],
 [[1666, 1557]],
 [[1667, 1558]],
 [[1717, 1558]],
 [[1718, 1559]],
 [[1776, 1559]],
 [[1777, 1560]],
 [[1804, 1560]],
 [[1805, 1561]],
 [[1835, 1561]],
 [[1836, 1562]],
 [[1916, 1562]],
 [[1916, 1559]],
 [[1909, 1559]],
 [[1908, 1558]],
 [[1880, 1558]],
 [[1879, 1557]],
 [[1847, 1557]],
 [[1846, 1556]],
 [[1794, 1556]],
 [[1793, 1555]],
 [[1763, 1555]],
 [[1762, 1554

In [33]:
contoursssss, _ = cv2.findContours(horizontal, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

In [34]:
len(contoursssss[0])

9

In [35]:
contoursssss[0]

array([[[1138, 1576]],

       [[1138, 1579]],

       [[1426, 1579]],

       [[1333, 1579]],

       [[1332, 1578]],

       [[1273, 1578]],

       [[1272, 1577]],

       [[1216, 1577]],

       [[1215, 1576]]], dtype=int32)

In [47]:
contours[1].tolist()

[[[36, 1534]],
 [[36, 1535]],
 [[36, 1536]],
 [[36, 1537]],
 [[36, 1538]],
 [[36, 1539]],
 [[37, 1539]],
 [[38, 1539]],
 [[39, 1539]],
 [[40, 1539]],
 [[41, 1539]],
 [[42, 1539]],
 [[43, 1539]],
 [[44, 1539]],
 [[45, 1539]],
 [[46, 1539]],
 [[47, 1539]],
 [[48, 1539]],
 [[49, 1539]],
 [[50, 1539]],
 [[51, 1539]],
 [[52, 1539]],
 [[53, 1539]],
 [[54, 1539]],
 [[55, 1539]],
 [[56, 1539]],
 [[57, 1539]],
 [[58, 1539]],
 [[59, 1539]],
 [[60, 1539]],
 [[61, 1539]],
 [[62, 1539]],
 [[63, 1539]],
 [[64, 1539]],
 [[65, 1539]],
 [[66, 1539]],
 [[67, 1539]],
 [[68, 1539]],
 [[69, 1539]],
 [[70, 1539]],
 [[71, 1539]],
 [[72, 1539]],
 [[73, 1539]],
 [[74, 1539]],
 [[75, 1539]],
 [[76, 1539]],
 [[77, 1539]],
 [[78, 1539]],
 [[79, 1539]],
 [[80, 1539]],
 [[81, 1539]],
 [[82, 1539]],
 [[83, 1539]],
 [[84, 1539]],
 [[85, 1539]],
 [[86, 1539]],
 [[87, 1539]],
 [[88, 1539]],
 [[89, 1539]],
 [[90, 1539]],
 [[91, 1539]],
 [[92, 1539]],
 [[93, 1539]],
 [[94, 1539]],
 [[95, 1539]],
 [[96, 1539]],
 [[97, 153

In [48]:
groups = []
# for contour in contours:
for item in contours[1]:
    print(item)
    # # print(item[0][])
    # y = item[0][1]

    # found = False
    # for group in groups:
    #     print(group[0][0])
    #     break
    #     if (group[0][0] == y).any():
    #         group.append(item)
    #         found = True
    #         break

    # if not found:
    #     groups.append([item])



[[  36 1534]]
[[  36 1535]]
[[  36 1536]]
[[  36 1537]]
[[  36 1538]]
[[  36 1539]]
[[  37 1539]]
[[  38 1539]]
[[  39 1539]]
[[  40 1539]]
[[  41 1539]]
[[  42 1539]]
[[  43 1539]]
[[  44 1539]]
[[  45 1539]]
[[  46 1539]]
[[  47 1539]]
[[  48 1539]]
[[  49 1539]]
[[  50 1539]]
[[  51 1539]]
[[  52 1539]]
[[  53 1539]]
[[  54 1539]]
[[  55 1539]]
[[  56 1539]]
[[  57 1539]]
[[  58 1539]]
[[  59 1539]]
[[  60 1539]]
[[  61 1539]]
[[  62 1539]]
[[  63 1539]]
[[  64 1539]]
[[  65 1539]]
[[  66 1539]]
[[  67 1539]]
[[  68 1539]]
[[  69 1539]]
[[  70 1539]]
[[  71 1539]]
[[  72 1539]]
[[  73 1539]]
[[  74 1539]]
[[  75 1539]]
[[  76 1539]]
[[  77 1539]]
[[  78 1539]]
[[  79 1539]]
[[  80 1539]]
[[  81 1539]]
[[  82 1539]]
[[  83 1539]]
[[  84 1539]]
[[  85 1539]]
[[  86 1539]]
[[  87 1539]]
[[  88 1539]]
[[  89 1539]]
[[  90 1539]]
[[  91 1539]]
[[  92 1539]]
[[  93 1539]]
[[  94 1539]]
[[  95 1539]]
[[  96 1539]]
[[  97 1539]]
[[  98 1539]]
[[  99 1539]]
[[ 100 1539]]
[[ 101 1539]]
[[ 102

In [42]:
print(groups[0].tolist())

AttributeError: 'list' object has no attribute 'tolist'

In [26]:
contours[0].tolist()

[[[1138, 1576]],
 [[1138, 1577]],
 [[1138, 1578]],
 [[1138, 1579]],
 [[1139, 1579]],
 [[1140, 1579]],
 [[1141, 1579]],
 [[1142, 1579]],
 [[1143, 1579]],
 [[1144, 1579]],
 [[1145, 1579]],
 [[1146, 1579]],
 [[1147, 1579]],
 [[1148, 1579]],
 [[1149, 1579]],
 [[1150, 1579]],
 [[1151, 1579]],
 [[1152, 1579]],
 [[1153, 1579]],
 [[1154, 1579]],
 [[1155, 1579]],
 [[1156, 1579]],
 [[1157, 1579]],
 [[1158, 1579]],
 [[1159, 1579]],
 [[1160, 1579]],
 [[1161, 1579]],
 [[1162, 1579]],
 [[1163, 1579]],
 [[1164, 1579]],
 [[1165, 1579]],
 [[1166, 1579]],
 [[1167, 1579]],
 [[1168, 1579]],
 [[1169, 1579]],
 [[1170, 1579]],
 [[1171, 1579]],
 [[1172, 1579]],
 [[1173, 1579]],
 [[1174, 1579]],
 [[1175, 1579]],
 [[1176, 1579]],
 [[1177, 1579]],
 [[1178, 1579]],
 [[1179, 1579]],
 [[1180, 1579]],
 [[1181, 1579]],
 [[1182, 1579]],
 [[1183, 1579]],
 [[1184, 1579]],
 [[1185, 1579]],
 [[1186, 1579]],
 [[1187, 1579]],
 [[1188, 1579]],
 [[1189, 1579]],
 [[1190, 1579]],
 [[1191, 1579]],
 [[1192, 1579]],
 [[1193, 1579]

In [36]:
print(contours[4])

[[[1914 1103]]

 [[1914 1109]]

 [[1913 1110]]

 [[1913 1203]]

 [[1912 1204]]

 [[1912 1307]]

 [[1911 1308]]

 [[1911 1487]]

 [[1910 1488]]

 [[1910 1579]]

 [[1914 1579]]

 [[1914 1557]]

 [[1915 1556]]

 [[1915 1474]]

 [[1916 1473]]

 [[1916 1274]]

 [[1917 1273]]

 [[1917 1103]]]


In [ ]:
if (img is None) :
    print(f"{image_path} this is incorrect no img found")
else:
   
    # converting img to gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    # cv2.imwrite(f'./output/opencv/gray{i}.jpeg', gray)

    # splits the image and clahe 
    # C – Contrast
    # L – Limited
    # A – Adaptive
    # H – Histogram
    # E – Equalization
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gray_clahe = clahe.apply(gray)    
    # cv2.imwrite(f'./output/opencv/gray_clahe{i}.jpeg', gray_clahe)
    
    # Better threshold (adaptive handles uneven lighting)
    thresh = cv2.adaptiveThreshold(
        gray_clahe, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15, 10
    )
     
    
    # --- Horizontal lines ---
    horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (60,1))
    horizontal = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, horizontal_kernel)
    # cv2.imwrite(f'./output/opencv/horizontal{i}.jpeg', horizontal)
    
    # connect broken horizontal lines
    horizontal = cv2.dilate(horizontal, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/horizontal_dilated{i}.jpeg', horizontal)
    
    # --- Vertical lines ---
    vertical_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1,60))
    vertical = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, vertical_kernel)
    # cv2.imwrite(f'./output/opencv/vertical{i}.jpeg', vertical)
     
    # connect broken vertical lines
    vertical = cv2.dilate(vertical, np.ones((2,2),np.uint8), iterations=2)
    # cv2.imwrite(f'./output/opencv/vertical_dilate{i}.jpeg', vertical)
    
    
    # Combine
    boxes = cv2.add(horizontal, vertical)
    # cv2.imwrite(f'./output/opencv/boxes_combine{i}.jpeg', boxes)
    
    # Final closing to join gaps
    kernel = np.ones((10,10), np.uint8)
    boxes = cv2.morphologyEx(boxes, cv2.MORPH_CLOSE, kernel, iterations=2)
    cv2.imwrite(f'./output/opencv/boxes_fill_gap{i}.jpeg', boxes)  
    # cv2.imwrite(f'./output/opencv/new/boxes_fill_gap{i}.jpeg', boxes)  

    # kernel = np.ones((2,2), np.uint8)
    # erode = cv2.erode(boxes, kernel, iterations=2)
    # cv2.imwrite(f'./output/opencv/erode{i}.jpeg', erode)
    
    # adding border
    h, w = img.shape[:2]
    border=cv2.rectangle(boxes, (0,0), (w,h), (255,255,255), 3)
    cv2.imwrite(f'./output/opencv/bordered{i}.jpeg', border)
    # cv2.imwrite(f'./output/opencv/new/bordered{i}.jpeg', border)
    
    # Find contours
    contours, _ = cv2.findContours(border, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    output = img.copy()
    cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)
    # cv2.imwrite(f'./output/opencv/new/output{i}.jpeg', output)

# find and draw the counter

In [ ]:
# output = img.copy()

# # for index,cnt in enumerate(contours):
# #     x,y,w,h = cv2.boundingRect(cnt)
# #     # filter noise
# #     if w > 100 and h > 100:
# cv2.drawContours(output, contours, -1, (0,255,0), 1)
# cv2.imwrite(f'./output/opencv/output{i}.jpeg', output)

In [ ]:
# contours

# Downscaling the paddle ocr border to small

#### ref code 

In [ ]:
# ref code 
# new_poly = []
# for poly in result[0]["rec_polys"]:
#     # convert to float for calculation
#     poly = np.array(poly, dtype=np.float32)
#     cv2.polylines(output, [np.array(poly, dtype=np.int32)],isClosed=True, color=(255, 0, 0),thickness=2) 

#     x_min, y_min = np.min(poly, axis=0)
#     x_max, y_max = np.max(poly, axis=0)
#     w = x_max - x_min
#     h = y_max - y_min

#     # adaptive shrink
#     shrink_factor_h = max(0.6, min(0.9, 1 - 20/w))
#     shrink_factor_v = max(0.6, min(0.9, 1 - 20/h))
#     # print(poly)
    
#     # compute center
#     center_x = np.mean(poly[:, 0])
#     center_y = np.mean(poly[:, 1])

#     poly_box=[]
#     for (x,y) in poly:
#         x_new=int(center_x+(x-center_x)*shrink_factor_h)
#         y_new=int(center_y+(y-center_y)*shrink_factor_v)
#         poly_box.append([x_new,y_new])
#     new_poly.append(np.array(poly_box, dtype=np.int16))
#     cv2.polylines(output, [np.array(poly_box, dtype=np.int32)],isClosed=True, color=(0, 255, 255),thickness=2) 
# cv2.imwrite(f'./output/opencv/scaling{i}.jpeg', output)

In [ ]:
def downscaling(poly_list):
    poly_downscaling=[]
    for poly in poly_list:
        # convert to float for calculation
        poly = np.array(poly, dtype=np.float32)
        cv2.polylines(output, [np.array(poly, dtype=np.int32)],isClosed=True, color=(255, 0, 0),thickness=2) 
    
        x_min, y_min = np.min(poly, axis=0)
        x_max, y_max = np.max(poly, axis=0)
        w = x_max - x_min
        h = y_max - y_min
    
        # adaptive shrink
        shrink_factor_h = max(0.6, min(0.9, 1 - 20/w))
        shrink_factor_v = max(0.6, min(0.9, 1 - 20/h))
        # print(poly)
        
        # compute center
        center_x = np.mean(poly[:, 0])
        center_y = np.mean(poly[:, 1])
    
        poly_box=[]
        for (x,y) in poly:
            x_new=int(center_x+(x-center_x)*shrink_factor_h)
            y_new=int(center_y+(y-center_y)*shrink_factor_v)
            poly_box.append([x_new,y_new])
        # cv2.polylines(output, [np.array(poly_box, dtype=np.int32)],isClosed=True, color=(0, 255, 255),thickness=2) 
        # cv2.imwrite(f'./output/opencv/scaling{i}.jpeg', output)
        poly_downscaling.append(np.array(poly_box, dtype=np.int16))
    return poly_downscaling


# make an array by combining the ploy points , text , and confident scores

In [ ]:
result[0]['rec_texts']

In [ ]:
downscaling_poly=downscaling(result[0]['dt_polys'])
ocr_result = [(poly , text , scores)
    for poly , text , scores  in 
        zip(downscaling_poly,
            result[0]['rec_texts'],
            result[0]['rec_scores'])]

In [ ]:
# downscaling_poly[0].tolist()

In [ ]:
ocr_result

# to get the overlapping percentage 

In [ ]:
# import numpy as np

# def rectangle_overlap_percentage(rectA, rectB ):
#     # """
#     # rectA and rectB: list of 4 points [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
#     # Can also be numpy arrays of shape (4,2)
#     # Returns overlap % relative to rectA
#     # """
#     # Convert to numpy arrays
#     rectA = np.array(rectA)
#     rectB = np.array(rectB)
    
#     # Get bounding box
#     x_min_A, x_max_A = rectA[:,0].min(), rectA[:,0].max()
#     y_min_A, y_max_A = rectA[:,1].min(), rectA[:,1].max()
    
#     x_min_B, x_max_B = rectB[:,0].min(), rectB[:,0].max()
#     y_min_B, y_max_B = rectB[:,1].min(), rectB[:,1].max()
    
#     # Overlap width & height
#     overlap_width = max(0, min(x_max_A, x_max_B) - max(x_min_A, x_min_B))
#     overlap_height = max(0, min(y_max_A, y_max_B) - max(y_min_A, y_min_B))
    
#     # Intersection area
#     intersection_area = overlap_width * overlap_height
    
#     # Area of rectangle A
#     area_A = (x_max_A - x_min_A) * (y_max_A - y_min_A)
    
#     # Overlap %
#     overlap_percentage = (intersection_area / area_A) * 100 if area_A > 0 else 0

#     return overlap_percentage

In [ ]:
# contours[1]

In [ ]:
# rectA = np.array((contours[1].reshape(-1,2)).tolist())
# rectA

In [ ]:
# result[0]['rec_polys'][8].tolist()

In [ ]:
# for ocr in ocr_result:
#     ocr[0]
# np.array(ocr[2])

In [ ]:
def rectangle_overlap_percentage(paddleocr_box, counters):
    #rectA = paddleocr text box
    #rectB = counter of the table
    # rectA and rectB: list of 4 points [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    #how much of a is in b
    # Convert to numpy arrays
    rectA_rectB_merge=[]
    counter=[]
    box=[]
    text=[]
    # for counter_i in range(len(counters)):
    for counter_poly in counters:
        ocr_box=[]
        ocr_text=[]
        # for paddleocr_i in range(len(paddleocr_box)):
        for paddleocr_poly in paddleocr_box:
            # rectA_org = paddleocr_box[paddleocr_i][0].tolist()
            # rectB_org = counters[counter_i][:, 0, :].tolist()
            
            # rectA_org = paddleocr_poly.tolist()
            # rectB_org = counter_poly.tolist()
                    
            # rectA = np.array(rectA_org)
            # rectB = np.array(rectB_org)
            
            rectA = np.array(paddleocr_poly[0].tolist())
            rectB = np.array((counter_poly.reshape(-1,2)).tolist())
            
            # Get bounding box
            x_min_A, x_max_A = rectA[:,0].min(), rectA[:,0].max()
            y_min_A, y_max_A = rectA[:,1].min(), rectA[:,1].max()
            
            x_min_B, x_max_B = rectB[:,0].min(), rectB[:,0].max()
            y_min_B, y_max_B = rectB[:,1].min(), rectB[:,1].max()
            
            # Overlap width & height
            overlap_width = max(0, min(x_max_A, x_max_B) - max(x_min_A, x_min_B))
            overlap_height = max(0, min(y_max_A, y_max_B) - max(y_min_A, y_min_B))
            
            # Intersection area
            intersection_area = overlap_width * overlap_height
            
            # Area of rectangle A
            area_A = (x_max_A - x_min_A) * (y_max_A - y_min_A)
            
            # Overlap %
            overlap_percentage = (intersection_area / area_A) * 100 if area_A > 0 else 0

            #append if overlapping is more 
            if(overlap_percentage > 90 and paddleocr_poly[2] > 0): #removed confident score to 0
                ocr_box.append(paddleocr_poly[0])
                ocr_text.append(paddleocr_poly[1])
                
                
            # else:
            #     continue
        if (ocr_box):
            # rectA_rectB_merge.append((counter_poly,ocr_box,ocr_text)) # adding [] to use it as counter point as an indiviudal points
            counter.append(counter_poly)
            box.append(ocr_box)
            text.append(ocr_text)
        # rectA_rectB_merge.append(ocr_box)
    return counter,box,text


In [ ]:
counter_only,box_only,text_only=rectangle_overlap_percentage(ocr_result,contours)

In [ ]:
len(counter_only[0])

In [ ]:
len(box_only[0])

In [ ]:
len(text_only[0])

In [ ]:
text_only

In [ ]:
# merged_counter_ocr[7][0] # gives counter

In [ ]:
# merged_counter_ocr[1][1] # gives poly of text

In [ ]:
# merged_counter_ocr[1][2] # gives text

In [ ]:
# len(contours)

In [ ]:
# len(grouped_counter_and_text)

In [ ]:
# ans[2][1]

In [ ]:
# np.array(grouped_counter_and_text[3][1]).tolist()

# display the counter and text poly box on the image [for testingg]

In [ ]:
img_test = cv2.imread(f'./output/opencv/output{i}.jpeg')
# for cnt_no in range(len(box_only)):
#     # cv2.drawContours(img_test, ans[cnt_no][1], -1, (255,0,0), 3)
#     for poly_ocr_box in box_only[cnt_no]:
#         cv2.polylines(img_test, [np.array(poly_ocr_box, dtype=np.int32)],isClosed=True, color=(255,255,0 ),thickness=3)
#     cv2.imwrite(f'./output/opencv/test{i}.jpeg', img_test)



cnt_no=1
cv2.drawContours(img_test, counter_only[cnt_no], -1, (255,0,255), 3)
for poly_ocr_box in box_only[cnt_no]:
    cv2.polylines(img_test, [np.array(poly_ocr_box, dtype=np.int32)],isClosed=True, color=(255,0,0 ),thickness=3)
cv2.imwrite(f'./output/opencv/test{i}.jpeg', img_test)


In [ ]:
# for i in range(len(contours)):
#     for j in range(len(result1[0]['rec_polys'])):
#         overlap = rectangle_overlap_percentage(result1[0]['rec_polys'][i].tolist(),contours[j][:, 0, :].tolist())
#         if(overlap > 50):
#             print(f"Overlap % of Rectangle A: {overlap:.2f}%")
#             cv2.drawContours(output, contours, j, (255,255,0), 2)
#             cv2.polylines(output, [np.array(result1[0]['rec_polys'][i], dtype=np.int32)],isClosed=True, color=(0, 0, 255),thickness=2)
# cv2.imwrite('./output/opencv/imagetry.jpeg', output)

# arranging the contours horizontally and vertically 

#### find the center of x and y

In [ ]:
def get_center_from_rect(rect):
    x, y, w, h = rect
    cx = x + w / 2
    cy = y + h / 2
    return cx, cy

#### grouping counter vertically 

In [ ]:
def group_contours_into_lines_h(contours, x_threshold=20):
    # Step 1: convert contours to bounding boxes 
    rects = [(cv2.boundingRect(cnt),cnt) for cnt in contours]
    #[x1,y1,w1,h1]

    # Step 2: compute centers
    rects_with_centers = [(rect, get_center_from_rect(rect),cnt) for rect,cnt in rects]
    #[[x1,y1,w1,h1],[cx,yc],[org cnt]]

    # Step 3: sort top → bottom
    rects_with_centers.sort(key=lambda r: r[1][0])  # sort by cx

    lines = []
    current_line = []

    for rect, (cx, cy),cnt in rects_with_centers:
        #if the current_line list is emptry append the first element 
        if not current_line:
            current_line.append((rect, cx, cy,cnt))
            continue
            
        #get the last cx
        _, prev_cx,_, _ = current_line[-1]

        # this is the buffer area where all belong to honrisontal 
        if abs(cx - prev_cx) < x_threshold:
            current_line.append((rect, cx, cy,cnt))
        #start as a new line 
        else:
            lines.append(current_line)
            current_line = [(rect, cx, cy,cnt)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line left → right
    result = []
    result_org = []
    for line in lines:
        line.sort(key=lambda r: r[2])  # sort by cy
        result.append([rect for rect, _, _,_ in line])
        result_org.append([cnt for _, _, _,cnt in line])

    return result,result_org
    # return lines

#### grouping counter horizontal

In [ ]:
def group_contours_into_lines_v(contours, y_threshold=20):
    # Step 1: convert contours to bounding boxes 
    rects = [(cv2.boundingRect(cnt),cnt) for cnt in contours]
    #[x1,y1,w1,h1]

    # Step 2: compute centers
    rects_with_centers = [(rect, get_center_from_rect(rect),cnt) for rect,cnt in rects]
    #[[x1,y1,w1,h1],[cx,yc],[org cnt]]

    # Step 3: sort top → bottom
    rects_with_centers.sort(key=lambda r: r[1][1])  # sort by cy

    lines = []
    current_line = []

    for rect, (cx, cy),cnt in rects_with_centers:
        #if the current_line list is emptry append the first element 
        if not current_line:
            current_line.append((rect, cx, cy,cnt))
            continue
            
        #get the last cy
        _, _, prev_cy,_ = current_line[-1]

        # this is the buffer area where all belong to honrisontal 
        if abs(cy - prev_cy) < y_threshold:
            current_line.append((rect, cx, cy,cnt))
        #start as a new line 
        else:
            lines.append(current_line)
            current_line = [(rect, cx, cy,cnt)]

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line left → right
    result = []
    result_org = []
    for line in lines:
        line.sort(key=lambda r: r[1])  # sort by cy
        result.append([rect for rect, _, _,_ in line])
        result_org.append([cnt for _, _, _,cnt in line])

    return result,result_org
    # return lines

In [ ]:
ans_h,ans_org_h = group_contours_into_lines_h(counter_only)
ans_v,ans_org_v = group_contours_into_lines_v(counter_only)

In [ ]:
for i_cnt in range(len(ans_org_h)):
    # output = cv2.imread(f'./output/opencv/output{i}.jpeg')
    # cv2.drawContours(output, ans_org_h[i_cnt], -1, (255,255,0), 3)
    # cv2.imwrite(f'./output/opencv/output{i_cnt}.jpeg', output)
    print(len(ans_org_h[i_cnt]))

In [ ]:
for i_cnt in range(len(ans_org_v)):
    # output = cv2.imread(f'./output/opencv/output{i}.jpeg')
    # cv2.drawContours(output, ans_org_v[i_cnt], -1, (255,255,0), 3)
    # cv2.imwrite(f'./output/opencv/output{i_cnt}.jpeg', output)
    print(len(ans_org_v[i_cnt]))

# getting the header of the table 

In [ ]:
def get_heading_col(number_col,ans_org_v):
    for i_cnt in range(len(ans_org_v)):
        if(len(ans_org_v[i_cnt]) == number_col):
            return ans_org_v[i_cnt]

In [ ]:
table_header=get_heading_col(6,ans_org_v)

In [ ]:
table_header

# match the table header with text and text box 

In [ ]:
def counter_matching (matching_counters,counter_only,box_only,text_only):
    box=[]
    text=[]
    for matching_counter in matching_counters:
        for index,counter in enumerate(counter_only):
            if (matching_counter.shape == counter.shape) and np.all(matching_counter == counter):
                box.append(box_only[index])
                text.append(text_only[index])
    return box,text
            
        

In [ ]:
matched_cnt=counter_matching(table_header,counter_only,box_only,text_only)

In [ ]:
matched_cnt[1]

In [ ]:
ans_org_v[0][1]

# header info extraction from vertical align 

In [ ]:
ans_org_v[0]

In [ ]:
def counter_matching_header(header_indexs,table_header,ans_org_h,counter_only,box_only,text_only):
    matched_cnt=[]
    for header_index in header_indexs:
        for index,counter_group in enumerate(ans_org_h):
            for counter in counter_group:
                if (table_header[header_index].shape == counter.shape) and np.all(table_header[header_index] == counter):
                    
                    matched_cnt.append(ans_org_h[index])

            #     box.append(box_only[index])
            #     text.append(text_only[index])
    return matched_cnt

In [ ]:
header_info=counter_matching_header([1,2,5],table_header,ans_org_h,counter_only,box_only,text_only)

In [ ]:
ref_info=counter_matching(header_info[0],counter_only,box_only,text_only)

In [ ]:
ref_info[1]

# arranging the text box of paddleocr horizontally

In [ ]:
def get_center(box):
    #[[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    x_coords = [p[0] for p in box]
    y_coords = [p[1] for p in box]
    return sum(x_coords) / 4, sum(y_coords) / 4


def group_boxes_into_lines(boxes, y_threshold=10):
    # Step 1: compute centers
    box_centers = [(box, get_center(box),text,score) for box,text,score in boxes]

    # Step 2: sort by Y (top to bottom)
    box_centers.sort(key=lambda b: b[1][1])

    lines = []
    current_line = []

    for box, (cx, cy) ,text ,score in box_centers:
        if score >= 0.9:
            if not current_line:
                current_line.append((box, cx, cy, text, score))
                continue
    
            _, _, prev_cy, _, _ = current_line[-1]
    
            # Step 3: check if same line
            if abs(cy - prev_cy) < y_threshold:
                current_line.append((box, cx, cy, text, score))
            else:
                lines.append(current_line)
                current_line = [(box, cx, cy, text, score)]
        else:
            continue

    if current_line:
        lines.append(current_line)

    # Step 4: sort each line by X (left to right)
    sorted_lines = []
    sorted_lines_text = []
    for line in lines:
        line.sort(key=lambda b: b[2])  # sort by center x
        sorted_lines.append([b[0] for b in line])
        sorted_lines_text.append([b[3] for b in line])

    return sorted_lines,sorted_lines_text

In [ ]:
poly,text = group_boxes_into_lines(ocr_result)

In [ ]:
# poly

In [ ]:
text[2]

In [ ]:
# ocr_result

# llm integration

In [ ]:
 result[0]['rec_texts']

In [ ]:
chr(10).join( result[0]['rec_texts'])

In [ ]:
# from llama_cpp import Llama

In [ ]:
llm = Llama(
    model_path="llama3-Q5_K_M.gguf",
    n_ctx=2048,
    n_threads=8,
    verbose=False  
)

In [ ]:
# prompt = f"""
# You are an expert receipt parser.

# Your task:
# Extract all products from OCR text.

# Rules:
# - A product may span multiple lines
# - Merge lines belonging to same item
# - Extract:
#   - product name
#   - quantity
#   - price
# - Ignore noise text
# - Extract total separately

# Return ONLY valid JSON like:

# {{
#   "items": [
#     {{"product": "", "quantity": "", "price": 0}}
#   ],
#   "total": 0
# }}

# OCR TEXT:
# {chr(10).join( result[0]['rec_texts'])}
# """

In [ ]:
prompt = f"""
You are a receipt extraction API.

Extract receipt items from OCR text.

Return ONLY valid JSON.
No markdown.
No explanation.
No extra text.
No notes.
No python code

JSON schema:
{{
  "items": [
    {{
      "product": "string",
      "quantity": "string",
      "price": 0
    }}
  ],
  "total_quantity": 0,
  "total_price": 0
}}

Rules:
- Extract ALL products
- Merge multiline products
- Ignore noise
- Extract quantity separately
- Extract total separately
- Price must be numeric

OCR:
{chr(10).join(result[0]['rec_texts'])}
"""

In [ ]:
# prompt = f"""
# You are a JSON API.

# Extract receipt items from OCR text.

# STRICT RULES:
# - Return ONLY ONE valid JSON object
# - Do NOT repeat output
# - Do NOT add markdown
# - Do NOT add explanations
# - Do NOT add text before or after JSON
# - Do NOT invent products
# - Ignore subtotal/total rows as products
# - Output must start with {{
# - Output must end with }}

# Schema:
# {{
#   "items": [
#     {{
#       "product": "string",
#       "quantity": "string",
#       "price": 0
#     }}
#   ],
#   "total_quantity": 0,
#   "total_price": 0
# }}

# OCR:
# {chr(10).join(result[0]['rec_texts'])}
# """

In [ ]:
# import json
# prompt = f"""
# You are a JSON API.

# Extract invoice items.

# Rules:
# - Ignore tax rows
# - Ignore totals
# - Merge multiline descriptions
# - Return valid JSON only

# Input:
# {json.dumps(rows, indent=2)}

# Return:
# {{
#   "items": [
#     {{
#       "product": "string",
#       "quantity": "string",
#       "price": 0
#     }}
#   ],
#   "total_quantity": 0,
#   "total_price": 0
# }}
# """

In [ ]:
output = llm(prompt, max_tokens=300)

In [ ]:
print(output["choices"][0]["text"])

In [ ]:
output